# L7 test errors and harmonization sensitivity

Run after notebooks 02 and 07. Baseline errors use `l1_cal_monthly.parquet` and the stored `l6_har06_factor.parquet`. Calibration remains 2018–2021 and test evaluation starts in 2022. No factor is fitted on test observations.

Grid and confidence sensitivities rebuild counts from paired processed files one year at a time. Variant factors use calibration months, and variants are compared over common comparable months. Defaults compare 0.1 and 0.25 degree grids and nominal/high, high-only and all known confidence classes.

HAR-12 reports `not_applicable` for the documented sensor inventory. A numerical transition analysis requires a separately defined transition and appropriate data. The L7 report documents the output contracts, which are not yet specified in the shared methodology.

Use the `metricas-pipeline` branch and enable a `GITHUB_TOKEN` Colab secret for private repository access. Real global VIIRS processing may need a high-memory runtime.


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import base64
import os
import subprocess
import sys

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Sprinklers')
REPO = Path('/content/repo')
environment = os.environ.copy()
try:
    token = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    token = None
if token:
    credential = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    environment.update(GIT_CONFIG_COUNT='1',
                       GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
                       GIT_CONFIG_VALUE_0=f'AUTHORIZATION: basic {credential}')
    del credential
if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], env=environment, check=True)
    subprocess.run(['git', '-C', str(REPO), 'switch', 'metricas-pipeline'], env=environment, check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'metricas-pipeline'],
                   env=environment, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'metricas-pipeline',
                    'https://github.com/Rui-Martins31/NASA_Space_Apps_2026_Sprinklers.git',
                    str(REPO)], env=environment, check=True)
del token, environment
sys.path.insert(0, str(REPO / 'data' / 'pipeline'))


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib
import json
import re
import subprocess
import pandas as pd
from metrics import harmonization_errors, harmonization_sensitivity, harmonization_transition

for module in [harmonization_errors, harmonization_sensitivity, harmonization_transition]:
    importlib.reload(module)
CAL_START = 2018
CAL_END = 2021
TEST_START = 2022
MIN_CAL_MONTHS = 24
MAX_UNKNOWN_DAYS = 0
YEARS = None  # Available processed years, used only for sensitivity.
FORCE = False
GRID_VARIANTS = harmonization_sensitivity.GRID_VARIANTS
CONFIDENCE_VARIANTS = harmonization_sensitivity.CONFIDENCE_VARIANTS
VARIANTS = dict(GRID_VARIANTS, **CONFIDENCE_VARIANTS)
GRID_BASELINE = 'grid_0.1'
CONFIDENCE_BASELINE = 'nominal_high'
PROCESSED = ROOT / 'processed'
METRICS = ROOT / 'metrics'
PARTIAL = METRICS / '_partial_lote7'
COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
PERIOD_OPTIONS = dict(cal_start=CAL_START, cal_end=CAL_END,
                      test_start=TEST_START, max_unknown_days=MAX_UNKNOWN_DAYS)
PARAMETERS = dict(PERIOD_OPTIONS, min_cal_months=MIN_CAL_MONTHS,
                  grid_variants=GRID_VARIANTS, confidence_variants=CONFIDENCE_VARIANTS,
                  years=YEARS, grid_baseline=GRID_BASELINE, confidence_baseline=CONFIDENCE_BASELINE)
PROCESSED_COLUMNS = ['country', 'instrument', 'satellite', 'latitude', 'longitude',
                     'acq_date', 'type', 'confidence_class']
MONTHLY_COLUMNS = ['country', 'year', 'month', 'series', 'cell_days', 'partial', 'n_unknown_days']

def atomic_parquet(frame, path):
    temporary = path.with_suffix('.parquet.tmp')
    frame.to_parquet(temporary, index=False)
    temporary.replace(path)

def atomic_json(value, path):
    temporary = path.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(value, sort_keys=True), encoding='utf-8')
    temporary.replace(path)


In [ ]:
monthly_path = METRICS / 'l1_cal_monthly.parquet'
factor_path = METRICS / 'l6_har06_factor.parquet'
missing = [str(path) for path in [monthly_path, factor_path] if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing L1/L6 inputs. Run notebooks 02 and 07 first: ' + ', '.join(missing))
monthly = pd.read_parquet(monthly_path, columns=MONTHLY_COLUMNS)
factors = pd.read_parquet(factor_path)
# Validate baseline inputs before processing larger raw files.
harmonization_errors.test_predictions(monthly, factors, **PERIOD_OPTIONS)
if YEARS is None:
    years = set()
    for path in PROCESSED.glob('*.parquet'):
        match = re.fullmatch(r'(modis|viirs-jpss1)_([0-9]{4})', path.stem)
        if match and int(match[2]) >= CAL_START:
            years.add(int(match[2]))
    years = sorted(years)
else:
    years = sorted({int(year) for year in YEARS if int(year) >= CAL_START})
if not years:
    raise FileNotFoundError(f'No selected processed sensor-year files in {PROCESSED}')
sources = {year: [PROCESSED / f'{sensor}_{year}.parquet'
                  for sensor in ['modis', 'viirs-jpss1']] for year in years}
missing = [str(path) for paths in sources.values() for path in paths if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing processed counterparts: ' + ', '.join(missing))
PARTIAL.mkdir(parents=True, exist_ok=True)
module_digest = hashlib.sha256(Path(harmonization_sensitivity.__file__).read_bytes()).hexdigest()


In [ ]:
monthly_parts = []
overlap_parts = []
for year in years:
    monthly_partial = PARTIAL / f'monthly_{year}.parquet'
    overlap_partial = PARTIAL / f'overlap_{year}.parquet'
    marker = PARTIAL / f'{year}.json'
    signature = dict(cal_start=CAL_START, variants=VARIANTS, code=module_digest, commit=COMMIT,
                     monthly_source=dict(size=monthly_path.stat().st_size, mtime_ns=monthly_path.stat().st_mtime_ns),
                     sources=[dict(name=path.name, size=path.stat().st_size, mtime_ns=path.stat().st_mtime_ns)
                              for path in sources[year]])
    signature = json.loads(json.dumps(signature, sort_keys=True))
    cached = None
    if not FORCE and monthly_partial.is_file() and overlap_partial.is_file() and marker.is_file():
        try:
            if json.loads(marker.read_text(encoding='utf-8')) == signature:
                cached = (pd.read_parquet(monthly_partial), pd.read_parquet(overlap_partial))
        except (OSError, ValueError):
            pass
    if cached is not None:
        monthly_parts.append(cached[0])
        overlap_parts.append(cached[1])
        print(f'Reusing sensitivity counts for {year}')
        continue
    marker.unlink(missing_ok=True)
    frames = []
    for instrument, path in zip(['MODIS', 'VIIRS'], sources[year]):
        frame = pd.read_parquet(path, columns=PROCESSED_COLUMNS)
        if not frame.instrument.astype('string').str.upper().eq(instrument).fillna(False).all():
            raise ValueError(f'Unexpected instrument in {path}')
        valid_dates = pd.to_datetime(frame.acq_date, errors='coerce', utc=True).dropna()
        if not valid_dates.dt.year.eq(year).all():
            raise ValueError(f'Acquisition dates outside filename year in {path}')
        frames.append(frame)
    detections = pd.concat(frames, ignore_index=True)
    del frames
    for column in ['country', 'instrument', 'satellite', 'confidence_class']:
        detections[column] = detections[column].astype('category')
    variant_monthly, variant_overlap = harmonization_sensitivity.build_variant_tables(
        detections, monthly.loc[monthly.year.eq(year)], VARIANTS, cal_start=CAL_START)
    atomic_parquet(variant_monthly, monthly_partial)
    atomic_parquet(variant_overlap, overlap_partial)
    atomic_json(signature, marker)
    monthly_parts.append(variant_monthly)
    overlap_parts.append(variant_overlap)
    del detections
    print(f'Processed sensitivity counts for {year}')
variant_monthly = pd.concat(monthly_parts, ignore_index=True)
variant_overlap = pd.concat(overlap_parts, ignore_index=True)


In [ ]:
grid_monthly = variant_monthly.loc[variant_monthly.variant.isin(GRID_VARIANTS)]
grid_overlap = variant_overlap.loc[variant_overlap.variant.isin(GRID_VARIANTS)]
confidence_monthly = variant_monthly.loc[variant_monthly.variant.isin(CONFIDENCE_VARIANTS)]
confidence_overlap = variant_overlap.loc[variant_overlap.variant.isin(CONFIDENCE_VARIANTS)]
tables = {
    'l7_har08_mae': harmonization_errors.har08_mae(monthly, factors, **PERIOD_OPTIONS),
    'l7_har09_wape': harmonization_errors.har09_wape(monthly, factors, **PERIOD_OPTIONS),
    'l7_har10_bias': harmonization_errors.har10_bias(monthly, factors, **PERIOD_OPTIONS),
    'l7_har11_error_reduction': harmonization_errors.har11_error_reduction(monthly, factors, **PERIOD_OPTIONS),
    'l7_har12_residual_step': harmonization_transition.har12_residual_step(monthly),
    'l7_har13_grid_sensitivity': harmonization_sensitivity.har13_grid_sensitivity(
        grid_monthly, grid_overlap, baseline=GRID_BASELINE, min_cal_months=MIN_CAL_MONTHS, **PERIOD_OPTIONS),
    'l7_har14_confidence_sensitivity': harmonization_sensitivity.har14_confidence_sensitivity(
        confidence_monthly, confidence_overlap, baseline=CONFIDENCE_BASELINE,
        min_cal_months=MIN_CAL_MONTHS, **PERIOD_OPTIONS),
}
log = []
executed_at = datetime.now(timezone.utc).isoformat()
for name, result in tables.items():
    atomic_parquet(result, METRICS / f'{name}.parquet')
    log.append(dict(metric=name, parameters=json.dumps(PARAMETERS, sort_keys=True),
                    sensitivity_years=json.dumps(years), commit=COMMIT, output_rows=len(result),
                    factor_source_sha256=hashlib.sha256(factor_path.read_bytes()).hexdigest(),
                    executed_at_utc=executed_at))
log_path = METRICS / '_log_lote7.csv'
temporary = log_path.with_suffix('.csv.tmp')
pd.DataFrame(log).to_csv(temporary, index=False)
temporary.replace(log_path)


In [ ]:
mae = tables['l7_har08_mae']
assert mae.n_adjusted_pairs.le(mae.n_pairs).all()
assert mae.mae_raw.dropna().ge(0).all()
assert tables['l7_har12_residual_step'].step_raw.isna().all()
for name, result in tables.items():
    print(name, result.shape)
    print(result.dtypes)
    print(result.head(3).to_string(index=False))
print(f'L7 completed: {len(tables)} tables written to {METRICS}')
